transformers → Provides pre-trained models and APIs to load, fine-tune, and run LLMs.

datasets → Easy access and processing of large datasets for training and evaluation.

peft → Enables parameter-efficient fine-tuning methods like LoRA without updating full model weights.

accelerate → Simplifies multi-GPU, mixed precision, and distributed training.

bitsandbytes → Enables low-bit quantization (4-bit/8-bit) to reduce memory usage.

trl → Provides tools like SFTTrainer for training LLMs using reinforcement learning and supervised fine-tuning.

In [1]:
!pip install -U transformers datasets peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.8/526.8 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer

In [16]:
# MODEL_NAME = "google/gemma-2-9b-it"
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = "./TinyLlama"

In [4]:
dataset = load_dataset("Abirate/english_quotes")

README.md: 0.00B [00:00, ?B/s]

quotes.jsonl:   0%|          | 0.00/647k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['quote', 'author', 'tags'],
        num_rows: 2508
    })
})

In [6]:
def format_example(example):
    quote = example["quote"].strip()
    tags = ", ".join(example["tags"]) if isinstance(example["tags"], list) else str(example["tags"])

    text = f"""<bos><start_of_turn>user
Generate relevant tags for the following quote.

Quote:
{quote}<end_of_turn>
<start_of_turn>model
{tags}<end_of_turn>"""
    return {"text": text}


In [7]:
train_dataset = dataset["train"].map(format_example)

Map:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [8]:
train_dataset

Dataset({
    features: ['quote', 'author', 'tags', 'text'],
    num_rows: 2508
})

#### This format teaches the model how to respond to a user instruction by mapping input text to the expected output.

In [14]:
print(train_dataset['text'][0])

<bos><start_of_turn>user
Generate relevant tags for the following quote.

Quote:
“Be yourself; everyone else is already taken.”<end_of_turn>
<start_of_turn>model
be-yourself, gilbert-perreira, honesty, inspirational, misattributed-oscar-wilde, quote-investigator<end_of_turn>


In [15]:
print(train_dataset['text'][1])

<bos><start_of_turn>user
Generate relevant tags for the following quote.

Quote:
“I'm selfish, impatient and a little insecure. I make mistakes, I am out of control and at times hard to handle. But if you can't handle me at my worst, then you sure as hell don't deserve me at my best.”<end_of_turn>
<start_of_turn>model
best, life, love, mistakes, out-of-control, truth, worst<end_of_turn>


In [17]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [18]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [37]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [20]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [21]:
model = get_peft_model(model, lora_config)

In [22]:
model.print_trainable_parameters()

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [28]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    bf16=True,
    fp16=False,
    report_to="none",
    remove_unused_columns=False,
)

In [29]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
)

In [30]:
trainer.train()

Step,Training Loss
10,2.411156
20,1.306427
30,1.110607
40,1.131837
50,1.167682
60,0.978026
70,0.968340
80,1.083410
90,1.184355
100,0.999656


TrainOutput(global_step=627, training_loss=1.0734855006946522, metrics={'train_runtime': 635.7394, 'train_samples_per_second': 3.945, 'train_steps_per_second': 0.986, 'total_flos': 1648821794881536.0, 'train_loss': 1.0734855006946522})

In [31]:
#saving lora adapter
model.save_pretrained(f"{OUTPUT_DIR}/adapter")

In [32]:
tokenizer.save_pretrained(f"{OUTPUT_DIR}/adapter")

('./TinyLlama/adapter/tokenizer_config.json',
 './TinyLlama/adapter/chat_template.jinja',
 './TinyLlama/adapter/tokenizer.json')

In [33]:
base_model=MODEL_NAME

In [35]:
base_model

'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

In [38]:
inference_model = PeftModel.from_pretrained(base_model, f"{OUTPUT_DIR}/adapter")

In [39]:
inference_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [40]:
prompt = """<bos><start_of_turn>user
Generate relevant tags for the following quote.

Quote:
If you want to achieve greatness, stop asking for permission.<end_of_turn>
<start_of_turn>model
"""

In [41]:
inputs = tokenizer(prompt, return_tensors="pt").to(inference_model.device)

In [42]:
with torch.no_grad():
    output = inference_model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [43]:
print(tokenizer.decode(output[0], skip_special_tokens=True))

<bos><start_of_turn>user
Generate relevant tags for the following quote.

Quote:
If you want to achieve greatness, stop asking for permission.<end_of_turn>
<start_of_turn>model
life, success<end_of_turn>


In [44]:
#saving full model
merged_model = inference_model.merge_and_unload()

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [45]:
tokenizer.save_pretrained("./merged_full_model")

('./merged_full_model/tokenizer_config.json',
 './merged_full_model/chat_template.jinja',
 './merged_full_model/tokenizer.json')

In [46]:
merged_model.save_pretrained("./merged_full_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]